# Module 08 — Panoptic Segmentation

Panoptic segmentation unifies instance (things) and semantic (stuff) segmentation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from panoptic_utils import panoptic_quality, merge_semantic_instance, encode_panoptic_id, decode_panoptic_id

## 1. Panoptic Quality Metric

PQ = SQ × RQ where SQ is average IoU of matched pairs and RQ is F1 score.

In [ ]:
# Synthetic example: 3 GT segments, 3 predictions
gt = np.zeros((100, 100), dtype=np.int64)
gt[10:50, 10:50] = 1  # person instance 1
gt[60:90, 20:60] = 2  # car instance
gt[0:100, 70:100] = 3 # sky (stuff)

pred = np.zeros((100, 100), dtype=np.int64)
pred[12:52, 12:52] = 1  # good overlap with GT 1
pred[65:92, 18:58] = 2  # good overlap with GT 2
pred[0:100, 72:100] = 3  # good overlap with GT 3
# Extra false positive
pred[0:15, 0:15] = 4

result = panoptic_quality(pred, gt)
for k, v in result.items():
    print(f'{k}: {v}')

## 2. HuggingFace Panoptic Pipeline

Using a pretrained Mask2Former for panoptic segmentation inference.

In [ ]:
from transformers import pipeline
from PIL import Image
import urllib.request, io

# Load panoptic segmentation pipeline
pipe = pipeline('image-segmentation', model='facebook/detr-resnet-50-panoptic')

URL = 'http://images.cocodataset.org/val2017/000000039769.jpg'
try:
    with urllib.request.urlopen(URL) as r: data = r.read()
    img = Image.open(io.BytesIO(data)).convert('RGB')
except:
    import numpy as np
    img = Image.fromarray(np.random.randint(50,200,(480,640,3),dtype=np.uint8))

results = pipe(img)
print(f'Found {len(results)} segments')
for r in results[:5]:
    print(f'  Label: {r["label"]}, score: {r["score"]:.3f}')

## Exercise — Merge Semantic + Instance Maps

Given a semantic map and an instance map from two separate models,
merge them into a single panoptic representation using `merge_semantic_instance`.

In [ ]:
### EXERCISE
# Create synthetic semantic and instance maps
semantic_map = np.zeros((128, 128), dtype=np.int64)
semantic_map[:64, :] = 1   # sky (stuff, id=1)
semantic_map[64:, :40] = 2  # road (stuff, id=2)
semantic_map[64:, 40:80] = 3  # person (thing, id=3)
semantic_map[64:, 80:] = 4   # car (thing, id=4)

instance_map = np.zeros((128, 128), dtype=np.int64)
instance_map[64:, 40:80] = 1  # person instance 1
instance_map[64:, 80:] = 1    # car instance 1

thing_ids = [3, 4]  # person and car are things

# TODO: call merge_semantic_instance and visualise the result
# panoptic_map, segments_info = merge_semantic_instance(semantic_map, instance_map, thing_ids)
# print('Segments:', segments_info)